In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv("../data/sim_1year.csv")
series = df["data_consumption_gb"].values


In [3]:
df.head()

,timestamp,data_consumption_gb
0,2023-01-01 00:00:00,70.944
1,2023-01-01 01:00:00,68.505
2,2023-01-01 02:00:00,73.612
3,2023-01-01 03:00:00,79.390
4,2023-01-01 04:00:00,77.357


In [5]:
n = len(series)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

In [8]:
from sklearn.preprocessing import StandardScaler

In [9]:
scaler = StandardScaler()
scaler.fit(series[:train_end].reshape(-1, 1))
scaled = scaler.transform(series.reshape(-1, 1)).ravel()

In [10]:
W = 24

In [12]:
import tensorflow as tf

In [13]:
def make_windows(arr, w):
    X = np.array([arr[i:i + w] for i in range(len(arr) - w)])
    y = arr[w:]
    return X[..., np.newaxis], y        

In [14]:
X_train, y_train = make_windows(scaled[:train_end], W)
X_val,   y_val   = make_windows(scaled[train_end - W:val_end], W)
X_test,  y_test_scaled = make_windows(scaled[val_end - W:], W)

In [15]:
print("Train/Val/Test:", X_train.shape, X_val.shape, X_test.shape)

Train/Val/Test: (6108, 24, 1) (1314, 24, 1) (1314, 24, 1)


In [16]:
tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(W, 1)),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

In [17]:
early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [18]:
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=64,
    callbacks=[early],
    verbose=2,
)

Epoch 1/50
96/96 - 10s - 107ms/step - loss: 0.3197 - mae: 0.3581 - val_loss: 0.2655 - val_mae: 0.2425
Epoch 2/50
96/96 - 4s - 40ms/step - loss: 0.1534 - mae: 0.2071 - val_loss: 0.2356 - val_mae: 0.2076
Epoch 3/50
96/96 - 5s - 52ms/step - loss: 0.1348 - mae: 0.1854 - val_loss: 0.2339 - val_mae: 0.2012
Epoch 4/50
96/96 - 4s - 38ms/step - loss: 0.1299 - mae: 0.1792 - val_loss: 0.2360 - val_mae: 0.2027
Epoch 5/50
96/96 - 4s - 39ms/step - loss: 0.1278 - mae: 0.1764 - val_loss: 0.2396 - val_mae: 0.2081
Epoch 6/50
96/96 - 4s - 39ms/step - loss: 0.1266 - mae: 0.1752 - val_loss: 0.2429 - val_mae: 0.2136
Epoch 7/50
96/96 - 4s - 39ms/step - loss: 0.1256 - mae: 0.1742 - val_loss: 0.2443 - val_mae: 0.2157
Epoch 8/50
96/96 - 4s - 40ms/step - loss: 0.1245 - mae: 0.1732 - val_loss: 0.2451 - val_mae: 0.2149


In [19]:
# 6) Test: GB'ye geri çevir ve naif baseline ile karşılaştır
y_pred = scaler.inverse_transform(model.predict(X_test).reshape(-1, 1)).ravel()
y_true = series[val_end:]

42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step


In [20]:
naive_pred = series[val_end - 1:-1]  # y_t = y_{t-1}

In [21]:
mae_rnn   = np.mean(np.abs(y_true - y_pred))
mae_naive = np.mean(np.abs(y_true - naive_pred))
print(f"RNN   MAE: {mae_rnn:.2f} GB")
print(f"Naive MAE: {mae_naive:.2f} GB")

RNN   MAE: 6.21 GB
Naive MAE: 6.55 GB


In [22]:
model.save("../model/telecom_trafic.keras")